# Predição de Salários em Tecnologia
### Análise Exploratória e Aprendizado de Máquina
**Autores:** Leonardo Uchôa, Mateus Brunozi, Rennã Samuel

---

## Estrutura do notebook
1. [Instalação e imports](#1-instalação-e-imports)
2. [Carregamento dos dados](#2-carregamento-dos-dados)
3. [Inspeção inicial](#3-inspeção-inicial)
4. [Análise Exploratória (EDA)](#4-análise-exploratória-eda)
5. [Pré-processamento correto](#5-pré-processamento-correto)
6. [Modelagem e treinamento](#6-modelagem-e-treinamento)
7. [Avaliação dos modelos](#7-avaliação-dos-modelos)
8. [Análise de resíduos](#8-análise-de-resíduos)
9. [Importância das features](#9-importância-das-features)
10. [Conclusões](#10-conclusões)


## 1. Instalação e Imports

In [ ]:
# Instala as bibliotecas necessárias caso não estejam presentes
!pip install -q seaborn scikit-learn xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

# Configurações visuais globais
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Imports OK')


## 2. Carregamento dos Dados

In [ ]:
#AJUSTE O CAMINHO CONFORME SEU AMBIENTE
# Opção A Google Colab (Drive montado):
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/job_salary_prediction_dataset.csv'

# Opção B Execução local:
DATA_PATH = 'job_salary_prediction_dataset.csv'
# 

df = pd.read_csv(DATA_PATH)

print(f'Shape do dataset: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
df.head()


## 3. Inspeção Inicial

In [ ]:
# Tipos de dados
# É fundamental entender quais colunas são numéricas e quais são categóricas
# antes de qualquer pré-processamento. Tratamentos diferentes serão aplicados
# a cada tipo.
print('=== Tipos de dados ===')
print(df.dtypes)

print('\n=== Valores nulos por coluna ===')
nulos = df.isnull().sum()
print(nulos[nulos >= 0])          # mostra todas as colunas

print('\n=== Duplicatas ===')
n_dup = df.duplicated().sum()
print(f'{n_dup} linhas duplicadas encontradas')


In [ ]:
# Estatísticas descritivas das variáveis numéricas
# O describe() resume média, desvio-padrão e percentis.
# Permite identificar rapidamente outliers (distância entre max e 75%) e
# se as distribuições estão equilibradas (mean ≈ median → simétrica).
df.describe().round(2)


In [ ]:
# Contagem de categorias por variável categórica 
# Verificamos se as categorias estão balanceadas.
# Desbalanceamento extremo pode indicar viés ou necessidade de estratificação.
cat_cols = ['job_title', 'education_level', 'industry',
            'company_size', 'location', 'remote_work']

for col in cat_cols:
    contagem = df[col].value_counts()
    print(f'\n[{col}] — {df[col].nunique()} categorias únicas')
    print(contagem.to_string())


## 4. Análise Exploratória (EDA)

A EDA tem como objetivo entender a distribuição das variáveis, identificar
padrões, relações com o target (salary) e possíveis problemas nos dados,
**antes** de qualquer modelagem.


In [ ]:
# Distribuição do target (salary)
# O histograma mostra a forma da distribuição.
# O boxplot evidencia mediana, quartis e outliers.
# Juntos, respondem: a distribuição é normal? Há outliers extremos?
# Precisamos transformar o target (ex: log) antes de modelar?

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['salary'], bins=60, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição do Salário')
axes[0].set_xlabel('Salário (USD)')
axes[0].set_ylabel('Frequência')

axes[1].boxplot(df['salary'], vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot do Salário')
axes[1].set_xlabel('Salário (USD)')

plt.tight_layout()
plt.savefig('results/distribuicao_salario.png', bbox_inches='tight')
plt.show()

media   = df['salary'].mean()
mediana = df['salary'].median()
std     = df['salary'].std()
skew    = df['salary'].skew()

print(f'Média:         ${media:,.0f}')
print(f'Mediana:       ${mediana:,.0f}')
print(f'Desvio-padrão: ${std:,.0f}')
print(f'Skewness:      {skew:.3f}')
print()
print('Interpretação do skewness:')
print('  < -0.5 → assimétrica à esquerda')
print('  entre -0.5 e 0.5 → aproximadamente normal')
print('  > 0.5  → assimétrica à direita (cauda longa)')
print(f'  → Neste caso ({skew:.3f}): leve assimetria à direita — sem necessidade de transformação log.')


In [ ]:
# Distribuição das features numéricas
# Verificamos se experience_years, skills_count e certifications seguem
# distribuições razoáveis. Distribuições uniformes ou muito assimétricas
# podem exigir tratamentos específicos.

num_cols = ['experience_years', 'skills_count', 'certifications']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    ax.hist(df[col], bins=30, color='teal', edgecolor='white', alpha=0.8)
    ax.set_title(f'Distribuição: {col}')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frequência')

plt.suptitle('Distribuição das Features Numéricas', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Mapa de correlação (apenas variáveis numéricas)
#
# IMPORTANTE: correlação de Pearson só é válida entre variáveis NUMÉRICAS reais.
# NÃO calculamos correlação com variáveis categóricas encodadas como inteiros
# (ex: location → 0,1,2,3...), pois isso implica falsamente que existe ordem
# e distância entre as categorias — o que não existe.
#
# Para variáveis categóricas, usaremos boxplots e testes estatísticos adequados
# na seção seguinte.

corr_df = df[num_cols + ['salary']].corr()

plt.figure(figsize=(6, 4))
sns.heatmap(corr_df, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Mapa de Correlação (variáveis numéricas)')
plt.tight_layout()
plt.savefig('results/correlacao.png', bbox_inches='tight')
plt.show()

print('Correlação de cada feature com salary:')
for col in num_cols:
    r = corr_df.loc[col, 'salary']
    print(f'  {col:20s}: {r:+.3f}  →  ', end='')
    if abs(r) > 0.4:
        print('correlação moderada-forte')
    elif abs(r) > 0.2:
        print('correlação fraca-moderada')
    else:
        print('correlação fraca')


In [ ]:
#Scatter: features numéricas vs salary
# Visualizamos a relação individual de cada feature numérica com o target.
# Permite ver se a relação é linear, não-linear ou inexistente.

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    ax.scatter(df[col], df['salary'], alpha=0.05, s=5, color='steelblue')
    ax.set_xlabel(col)
    ax.set_ylabel('Salary (USD)')
    ax.set_title(f'{col} vs Salary')

plt.suptitle('Relação entre Features Numéricas e Salário', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
#Salário por variável categórica (boxplots)
#
# Para entender o impacto das variáveis categóricas no salário, usamos boxplots.
# Cada caixa mostra a mediana, IQR e outliers de salary dentro de cada categoria.
# Esta é a análise correta — NÃO correlação de Pearson com inteiros encodados.
#
# Ordenamos pelo salário mediano para facilitar a comparação visual.

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ordem = df.groupby(col)['salary'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='salary', order=ordem,
                ax=axes[i], palette='muted')
    axes[i].set_title(f'Salário por {col}')
    axes[i].tick_params(axis='x', rotation=30)
    axes[i].set_xlabel('')

plt.suptitle('Distribuição do Salário por Variável Categórica', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('results/salario_por_categorias.png', bbox_inches='tight')
plt.show()


In [ ]:
# Análise cruzada: remote_work × education_level
# Responde diretamente à pergunta de pesquisa:
# "Trabalho remoto e nível de educação interagem no salário?"
# Um gráfico de barras agrupadas compara a média salarial para cada
# combinação de modalidade de trabalho e nível de educação.

pivot = df.groupby(['remote_work', 'education_level'])['salary'].mean().unstack()
pivot.plot(kind='bar', figsize=(10, 5), colormap='tab10', edgecolor='white')
plt.title('Salário Médio: Modalidade de Trabalho × Nível de Educação')
plt.xlabel('Tipo de Trabalho Remoto')
plt.ylabel('Salário Médio (USD)')
plt.xticks(rotation=0)
plt.legend(title='Educação', bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

# Verificação estatística básica: as médias de salary diferem entre grupos?
print('Salário médio por tipo de trabalho remoto:')
print(df.groupby('remote_work')['salary'].agg(['mean','median','std']).round(0))


## 5. Pré-processamento Correto

### Por que não usar Label Encoding em todas as variáveis?

O **Label Encoding** transforma categorias em inteiros (ex: `Australia→0`, `Canada→1`, `Germany→2`).
Isso funciona **somente** para variáveis **ordinais**, onde a ordem entre as categorias é real e significativa.

Exemplos válidos de Label Encoding:
- `education_level`: High School < Bachelor < Master < PhD (há hierarquia real)
- `company_size`: Startup < Small < Medium < Large < Enterprise (há hierarquia real)

Para variáveis **nominais** (sem ordem), como `location`, `job_title`, `industry` e `remote_work`,
usar Label Encoding introduz ordem artificial: o modelo passa a interpretar que
`Germany (2)` está "mais perto" de `Canada (1)` do que de `India (3)`, o que é
matematicamente sem sentido. Isso prejudica especialmente a Regressão Linear.

A solução correta para variáveis nominais é o **One-Hot Encoding**, que cria uma
coluna binária (0 ou 1) para cada categoria, sem implicar qualquer ordem.


In [ ]:
import os
os.makedirs('results', exist_ok=True)

# Definição dos tipos de variável 

# Variáveis ORDINAIS: possuem ordem natural e significativa.
# Usamos Label Encoding com mapeamento explícito para garantir a ordem correta.
ordinal_cols = ['education_level', 'company_size']

# Variáveis NOMINAIS: não possuem ordem. Usamos One-Hot Encoding.
nominal_cols = ['job_title', 'industry', 'location', 'remote_work']

# Variáveis numéricas: usadas diretamente, sem transformação.
numeric_cols = ['experience_years', 'skills_count', 'certifications']

# Target
TARGET = 'salary'

# Label Encoding manual para variáveis ordinais 
# Definimos explicitamente a ordem correta de cada nível.
# Isso evita que o LabelEncoder atribua ordens aleatórias (ordem alfabética).

edu_order = ['High School', 'Diploma', 'Bachelor', 'Master', 'PhD']
size_order = ['Startup', 'Small', 'Medium', 'Large', 'Enterprise']

edu_map  = {v: i for i, v in enumerate(edu_order)}
size_map = {v: i for i, v in enumerate(size_order)}

df_ml = df.copy()
df_ml['education_level'] = df_ml['education_level'].map(edu_map)
df_ml['company_size']    = df_ml['company_size'].map(size_map)

print('Mapeamento education_level:', edu_map)
print('Mapeamento company_size:   ', size_map)

# One-Hot Encoding para variáveis nominais 
# drop='first' elimina uma categoria por variável para evitar multicolinearidade
# (dummy variable trap): se temos N categorias, N-1 colunas são suficientes.
df_ml = pd.get_dummies(df_ml, columns=nominal_cols, drop_first=True, dtype=int)

X = df_ml.drop(columns=TARGET)
y = df_ml[TARGET]

print(f'\nShape final após encoding: {X.shape}')
print(f'  → {len(numeric_cols)} features numéricas originais')
print(f'  → {len(ordinal_cols)} features ordinais encodadas')
print(f'  → {X.shape[1] - len(numeric_cols) - len(ordinal_cols)} features geradas pelo One-Hot Encoding')
print(f'\nPrimeiras colunas do dataset pré-processado:')
print(list(X.columns[:12]), '...')


## 6. Modelagem e Treinamento

### Modelos comparados
| Modelo | Tipo | Justificativa |
|---|---|---|
| Regressão Linear | Baseline | Simples, interpretável, sensível a encoding incorreto |
| Random Forest | Ensemble (bagging) | Captura não-linearidades, robusto a outliers |
| XGBoost | Ensemble (boosting) | Estado da arte em dados tabulares estruturados |

### Estratégia de avaliação: Validação Cruzada (K-Fold)
Um único split 80/20 pode favorecer ou prejudicar um modelo por acaso.
A **validação cruzada com K=5** divide os dados em 5 partes, treina em 4
e testa na 1ª restante, repetindo o processo 5 vezes. A métrica final
é a **média das 5 rodadas**, muito mais confiável estatisticamente.


In [ ]:
# Divisão treino/teste
# Mantemos um conjunto de teste separado para avaliação final.
# O K-Fold será aplicado APENAS sobre o conjunto de treino para seleção de modelo.
# O teste só é usado uma vez, no final — isso evita data leakage.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Treino: {X_train.shape[0]:,} amostras')
print(f'Teste:  {X_test.shape[0]:,} amostras')


In [ ]:
# Definição dos modelos
models = {
    'Regressão Linear': LinearRegression(),
    'Random Forest':    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'XGBoost':          XGBRegressor(n_estimators=100, random_state=42,
                                     seed=42, verbosity=0),
}

# Validação cruzada (K=5) sobre o conjunto de treino
# scoring='neg_root_mean_squared_error': sklearn usa valores negativos para
# métricas de erro, pois maximiza internamente. Usamos valor absoluto ao exibir.

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print('Validação cruzada (5-fold) — RMSE por fold:')
print('-' * 60)

for name, model in models.items():
    scores = cross_val_score(
        model, X_train, y_train,
        cv=kf,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    rmse_scores = -scores   # converte de negativo para positivo
    cv_results[name] = rmse_scores
    print(f'{name:25s} | média: ${rmse_scores.mean():,.0f} ± ${rmse_scores.std():,.0f}')

print()
print('Interpretação: menor RMSE = melhor modelo.')
print('O intervalo ± indica a variação entre os folds — quanto menor, mais estável.')


In [ ]:
# Treinamento final no conjunto completo de treino
# Após validação cruzada (seleção de modelo), treinamos cada modelo
# no X_train completo e avaliamos no X_test (que nunca foi visto antes).

trained_models = {}
final_results  = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae  = mean_absolute_error(y_test, preds)
    r2   = r2_score(y_test, preds)

    trained_models[name] = (model, preds)
    final_results.append({'Modelo': name, 'RMSE (USD)': rmse,
                          'MAE (USD)': mae, 'R²': r2})

    print(f'{name:25s} | RMSE: ${rmse:,.0f} | MAE: ${mae:,.0f} | R²: {r2:.4f}')

results_df = pd.DataFrame(final_results).set_index('Modelo')


## 7. Avaliação dos Modelos

### Métricas utilizadas
- **RMSE** (Root Mean Squared Error): penaliza erros maiores com mais peso. Unidade: USD.
- **MAE** (Mean Absolute Error): erro médio absoluto em dólares — mais intuitivo.
- **R²**: percentual da variância do salário explicado pelo modelo (1.0 = perfeito).


In [ ]:
#Tabela comparativa
print('=== Resultados no conjunto de teste ===')
print(results_df.to_string())

#Gráfico comparativo 
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
cores = ['#4c72b0', '#55a868', '#c44e52']

for ax, metric in zip(axes, ['RMSE (USD)', 'MAE (USD)', 'R²']):
    bars = ax.bar(results_df.index, results_df[metric], color=cores)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, results_df[metric]):
        label = f'${val:,.0f}' if metric != 'R²' else f'{val:.4f}'
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.01, label,
                ha='center', fontsize=8)

plt.suptitle('Comparação de Modelos — Conjunto de Teste', fontsize=13)
plt.tight_layout()
plt.savefig('results/comparacao_modelos.png', bbox_inches='tight')
plt.show()


In [ ]:
# Boxplot dos RMSEs da validação cruzada
# Permite visualizar não só a média, mas a VARIABILIDADE de cada modelo
# entre os 5 folds. Um modelo com média boa mas alta variação é menos confiável.

cv_df = pd.DataFrame(cv_results)

plt.figure(figsize=(8, 4))
cv_df.boxplot(column=list(cv_results.keys()))
plt.title('Distribuição do RMSE — Validação Cruzada (5 folds)')
plt.ylabel('RMSE (USD)')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

print('Estabilidade entre os folds (desvio-padrão do RMSE):')
for name, scores in cv_results.items():
    print(f'  {name:25s}: ±${scores.std():,.0f}')


## 8. Análise de Resíduos

A análise de resíduos é **obrigatória** em qualquer problema de regressão.
Ela responde: o modelo erra de forma aleatória (ideal) ou há padrões nos erros?

- **Predito vs. Real**: pontos devem se alinhar à diagonal. Desvios sistemáticos
  indicam que o modelo não captura algum padrão.
- **Histograma dos resíduos**: idealmente normal e centrado em zero.
  Assimetria indica erros sistemáticos (o modelo sub ou superestima sistematicamente).


In [ ]:
# Análise de resíduos para cada modelo 
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
cores_modelo = {'Regressão Linear': '#4c72b0',
                'Random Forest':    '#55a868',
                'XGBoost':          '#c44e52'}

for col_idx, (name, (model, preds)) in enumerate(trained_models.items()):
    residuos = y_test.values - preds
    cor = cores_modelo[name]

    # Linha superior: predito vs. real
    ax_top = axes[0, col_idx]
    ax_top.scatter(preds, y_test, alpha=0.1, s=4, color=cor)
    # Linha de referência: predição perfeita (y = x)
    lim = [min(preds.min(), y_test.min()), max(preds.max(), y_test.max())]
    ax_top.plot(lim, lim, 'k--', linewidth=1, label='Predição perfeita')
    ax_top.set_xlabel('Predito (USD)')
    ax_top.set_ylabel('Real (USD)')
    ax_top.set_title(f'{name}\nPredito vs. Real')
    ax_top.legend(fontsize=8)

    # Linha inferior: distribuição dos resíduos
    ax_bot = axes[1, col_idx]
    ax_bot.hist(residuos, bins=60, color=cor, edgecolor='white', alpha=0.8)
    ax_bot.axvline(0, color='black', linewidth=1, linestyle='--')
    ax_bot.set_xlabel('Resíduo (Real − Predito) em USD')
    ax_bot.set_ylabel('Frequência')
    ax_bot.set_title(f'Distribuição dos Resíduos\n'
                     f'Média: ${residuos.mean():+,.0f} | Std: ${residuos.std():,.0f}')

plt.suptitle('Análise de Resíduos por Modelo', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('results/residuos.png', bbox_inches='tight')
plt.show()

print('Erro médio (bias) de cada modelo — idealmente próximo de zero:')
for name, (model, preds) in trained_models.items():
    bias = (y_test.values - preds).mean()
    print(f'  {name:25s}: ${bias:+,.0f}')


## 9. Importância das Features

Analisamos a importância das features nos dois melhores modelos
(Random Forest e XGBoost) para comparar quais variáveis cada um considera relevantes.

**Random Forest** usa a redução média de impureza (Mean Decrease Impurity).
**XGBoost** usa o critério `gain` (ganho médio de informação quando a feature é usada em um split).


In [ ]:
# Importância das features: Random Forest e XGBoost
rf_model   = trained_models['Random Forest'][0]
xgb_model  = trained_models['XGBoost'][0]

# Importância pelo Random Forest (mean decrease impurity)
rf_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

# Importância pelo XGBoost (gain = contribuição média para redução de erro)
xgb_imp = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)

# Exibimos as top-15 features de cada modelo
TOP_N = 15

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

rf_imp.head(TOP_N).sort_values().plot(kind='barh', ax=axes[0], color='#55a868')
axes[0].set_title(f'Top {TOP_N} Features — Random Forest\n(Mean Decrease Impurity)')
axes[0].set_xlabel('Importância')

xgb_imp.head(TOP_N).sort_values().plot(kind='barh', ax=axes[1], color='#c44e52')
axes[1].set_title(f'Top {TOP_N} Features — XGBoost\n(Feature Importance — Gain)')
axes[1].set_xlabel('Importância')

plt.tight_layout()
plt.savefig('results/feature_importance.png', bbox_inches='tight')
plt.show()

# Tabela comparativa das top-10
print('Top 10 features mais importantes:')
comp = pd.DataFrame({
    'Random Forest': rf_imp.head(10),
    'XGBoost':       xgb_imp.head(10)
}).fillna(0).round(4)
print(comp.to_string())


## 10. Conclusões

Esta seção responde diretamente às **4 perguntas de pesquisa** definidas no início do projeto.


In [ ]:
# Resposta às perguntas de pesquisa

print('=' * 65)
print('RESPOSTAS ÀS PERGUNTAS DE PESQUISA')
print('=' * 65)

print('''
1. QUAIS FATORES MAIS INFLUENCIAM O SALÁRIO?
   → Localização (país) é o fator dominante (~33% de importância no RF).
   → Experiência (anos) é o segundo fator mais relevante (~20%).
   → Porte da empresa e cargo têm impacto similar (~16-17% cada).
   → Skills, certificações, setor e modalidade remota têm
     influência marginal individualmente.

2. EXPERIÊNCIA PESA MAIS QUE EDUCAÇÃO?
   → Sim. experience_years tem correlação de Pearson com salary = 0.438,
     enquanto education_level (ordinal) tem correlação bem menor.
   → Na importância das features, experiência supera educação
     consistentemente em ambos os modelos (RF e XGBoost).

3. TRABALHO REMOTO IMPACTA A REMUNERAÇÃO?
   → NÃO, de forma relevante. Os boxplots mostram distribuições
     de salário quase idênticas entre os três tipos (Yes/No/Hybrid).
   → remote_work aparece como uma das features menos importantes
     nos dois modelos — sua contribuição é marginal.

4. QUAL MODELO DE ML PERFORMA MELHOR?
   → XGBoost superou os demais em todas as métricas no conjunto de teste:
''')

print(results_df.to_string())

print('''
   → A validação cruzada confirma a superioridade do XGBoost e
     do Random Forest sobre a Regressão Linear.
   → A Regressão Linear performa pior em parte porque relações
     não-lineares entre features e salário não são capturadas.

LIMITAÇÕES DO ESTUDO:
   • Dataset sintético e artificialmente balanceado — o R² elevado
     reflete padrões artificiais; resultados podem não generalizar
     para dados reais de mercado.
   • Hiperparâmetros padrão foram usados (n_estimators=100).
     Uma busca com GridSearchCV poderia melhorar os modelos.
   • SHAP values poderiam fornecer interpretabilidade mais precisa
     que a importância padrão do Random Forest.
''')


---
## Próximos passos sugeridos

1. **Testar com dados reais** (ex: Stack Overflow Survey, Levels.fyi) para validar generalização.
2. **Tuning de hiperparâmetros** com `RandomizedSearchCV` para RF e XGBoost.
3. **SHAP values** para interpretabilidade global e local dos modelos.
4. **Target Encoding** como alternativa ao One-Hot para variáveis com muitas categorias.
5. **Stacking** dos modelos como experimento adicional de ensemble.
